# Multi-task output visualisation

This notebook scans an array-job output directory and plots key outputs for each successfully completed task.

For each completed task (must contain both `details.yaml` and `idata.nc`), it displays:

1. Calibration (Figure 1 style from manuscript)
2. Posterior vs prior distributions
3. Diff outputs using `plot_diff_outputs()`
4. Four-panel projected trajectories for baseline vs `scenario_3`

In [ ]:
from pathlib import Path
from math import ceil
import re
import yaml

import arviz as az
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from IPython.display import display, Markdown

import tbh.plotting as pl
import tbh.runner_tools as rt
from tbh.model import get_tb_model
from tbh.paths import REPO_ROOT_PATH
from estival.model import BayesianCompartmentalModel

plt.style.use("ggplot")

In [ ]:
# Set the array-job output directory to scan
base_dir = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "58681106_array_job"

if not base_dir.exists():
    raise FileNotFoundError(f"Directory not found: {base_dir}")

print("Scanning:", base_dir)

In [ ]:
task_dirs

In [ ]:
task_dirs = sorted(
    [p for p in base_dir.iterdir() if p.is_dir()]
)

if not task_dirs:
    raise ValueError(f"No task folders found in {base_dir}")

records = []
for task_dir in task_dirs:
    files = [x for x in task_dir.iterdir() if x.is_file()]
    names = {f.name for f in files}

    has_idata = "idata.nc" in names
    has_details = "details.yaml" in names

    if len(files) == 0:
        status = "empty"
    elif has_idata and has_details:
        status = "completed"
    elif has_idata:
        status = "partial"
    else:
        status = "failed"

    records.append({
        "task": task_dir.name,
        "task_path": task_dir,
        "n_files": len(files),
        "status": status,
    })

status_df = pd.DataFrame(records)
display(status_df[["task", "n_files", "status"]])
display(status_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

completed_df = status_df.loc[status_df["status"] == "completed"].copy()
print(f"Completed tasks: {len(completed_df)}")

In [ ]:
config_map_path = base_dir / "task_config_map.yaml"
if not config_map_path.exists():
    raise FileNotFoundError(f"Config map not found: {config_map_path}")

with open(config_map_path, "r") as f:
    raw_task_config_map = yaml.safe_load(f)

config_source = raw_task_config_map.get("tasks", raw_task_config_map) if isinstance(raw_task_config_map, dict) else raw_task_config_map
task_to_config = config_source.get("task_to_config", {}) if isinstance(config_source, dict) else {}

def get_task_config(task_name):
    task_num = int(task_name.split("_")[1])
    return task_to_config.get(task_num, task_to_config.get(str(task_num)))

def short_config_label(task_cfg):
    if not isinstance(task_cfg, dict):
        return "cfg_unknown"

    rel_sus = task_cfg.get("rel_sus_unreachable", "na")
    reg = task_cfg.get("clinical_regression_rate", "na")
    prog = task_cfg.get("clinical_progression_rate", "na")
    return f"rel_sus_{rel_sus} reg_{reg} prog_{prog}"

completed_df["task_config"] = completed_df["task"].apply(get_task_config)
completed_df["config_label"] = completed_df["task_config"].apply(short_config_label)
display(completed_df[["task", "config_label"]])

In [ ]:
def make_figure_1(uncertainty_df, bcm, colour="#B22222"):
    selected_outputs = [
        "pearl_posXreach_reachable_per100k",
        "cxr_posXreach_reachable_per100k",
        "perc_prev_subclinicalXreach_reachable",
        "perc_prev_infectiousXreach_reachable",
        "notifications",
    ]

    n_col = 3
    n_panels = len(selected_outputs) + 1
    n_row = ceil(n_panels / n_col)

    fig, axes = plt.subplots(n_row, n_col, figsize=(5 * n_col, 3.6 * n_row))
    axes = axes.flatten()

    for i, output in enumerate(selected_outputs):
        ax = axes[i]
        x_min = 1990 if output == "notifications" else 2010
        pl.plot_model_fit_with_uncertainty(ax, uncertainty_df, output, bcm, x_lim=(x_min, 2025), colour=colour)
        if i == 0:
            ax.legend()

    ax = axes[len(selected_outputs)]
    agegroups = ["3_9", "10", "15+", "18+"]
    model_median, model_low, model_high, observed, x_tick_labels = [], [], [], [], []

    for age in agegroups:
        output_name = f"tst_posXage_{age}Xreach_reachable_perc"
        year = bcm.targets[output_name].data.index[0]
        q = uncertainty_df[output_name].loc[year]
        obs = bcm.targets[output_name].data.iloc[0]

        model_median.append(q["0.5"])
        model_low.append(q["0.025"])
        model_high.append(q["0.975"])
        observed.append(obs)

        suffix = f" y.o.\n({year})"
        if age == "3_9":
            x_tick_labels.append("3-9" + suffix)
        elif age == "15+":
            x_tick_labels.append("15+" + suffix)
        else:
            x_tick_labels.append(f"{age}" + suffix)

    x = range(len(agegroups))
    ax.errorbar(
        [i - 0.06 for i in x],
        model_median,
        yerr=[
            [m - l for m, l in zip(model_median, model_low)],
            [h - m for h, m in zip(model_high, model_median)],
        ],
        fmt="D",
        color=colour,
        ecolor=colour,
        markersize=4,
        elinewidth=2.0,
        capsize=0,
        label="Model (median, 95% CI)",
    )
    ax.scatter([i + 0.06 for i in x], observed, color="black", s=15, zorder=5, label="Observed")
    ax.set_xticks(list(x))
    ax.set_xticklabels(x_tick_labels)
    ax.set_ylabel(pl.title_lookup["tst_posXreach_reachable_perc"])
    ax.grid(alpha=0.3)

    model_handle = mlines.Line2D([], [], color=colour, marker="D", markersize=4, linestyle="-", label="Model (median, 95% CI)")
    obs_handle = mlines.Line2D([], [], color="black", marker="o", linestyle="None", markersize=4, label="Observed")
    ax.legend(handles=[obs_handle, model_handle], frameon=False, loc="best")

    for j in range(n_panels, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    return fig

In [ ]:
params, priors, tv_params = rt.get_parameters_and_priors()
intervention_scenarios = [sc.sc_id for sc in rt.SCENARIOS]
trajectory_outputs = [
    "tb_incidence_per100k",
    "viable_tbi_prevalence_perc",
    "tb_prevalence_per100k",
    "tb_mortality_per100k",
]

for _, row in completed_df.iterrows():
    task_name = row["task"]
    task_path = Path(row["task_path"])
    cfg_label = row["config_label"]

    display(Markdown("---"))
    display(Markdown(f"## {task_name}"))
    display(Markdown(f"Config: `{cfg_label}`"))

    required = [task_path / "idata.nc", task_path / "details.yaml", task_path / "uncertainty_df_baseline.parquet"]
    if not all(p.exists() for p in required):
        missing = [str(p.name) for p in required if not p.exists()]
        display(Markdown(f"Skipping {task_name}: missing files {missing}"))
        continue

    idata = az.from_netcdf(task_path / "idata.nc")

    with open(task_path / "details.yaml", "r") as f:
        docs = list(yaml.safe_load_all(f))

    model_config = docs[1]
    analysis_config = docs[2]

    model = get_tb_model(model_config, tv_params)
    bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)

    available_unc = [p.name.replace("uncertainty_df_", "").replace(".parquet", "") for p in task_path.glob("uncertainty_df_*.parquet")]
    unc_dfs = {sc: pd.read_parquet(task_path / f"uncertainty_df_{sc}.parquet") for sc in available_unc}

    available_diff = [p.name.replace("diff_quantiles_df_ref_baseline_", "").replace(".parquet", "") for p in task_path.glob("diff_quantiles_df_ref_baseline_*.parquet")]
    diff_outputs_dfs = {sc: pd.read_parquet(task_path / f"diff_quantiles_df_ref_baseline_{sc}.parquet") for sc in available_diff}

    # 1) Calibration (Figure 1 style)
    if "baseline" in unc_dfs:
        fig = make_figure_1(unc_dfs["baseline"], bcm)
        fig.suptitle(f"{task_name} | {cfg_label}", y=1.02, fontsize=12)
        display(fig)
        plt.close(fig)
    else:
        display(Markdown("Calibration skipped: baseline uncertainty file missing."))

    # 2) Posterior distributions
    fig = pl.plot_post_prior_comparison(
        idata,
        analysis_config["burn_in"],
        req_vars=list(bcm.priors.keys()),
        priors=list(bcm.priors.values()),
    )
    fig.suptitle(f"{task_name} | {cfg_label}", y=1.02, fontsize=12)
    display(fig)
    plt.close(fig)

    # 3) Diff outputs from plot_diff_outputs()
    scenarios_for_diff = [sc for sc in intervention_scenarios if sc in diff_outputs_dfs]
    if scenarios_for_diff:
        for output in ["TB_averted_relative", "deaths_averted_relative"]:
            fig, ax = plt.subplots(1, 1, figsize=(0.6 * len(scenarios_for_diff), 4.0))
            pl.plot_diff_outputs(ax, diff_outputs_dfs, output, scenarios_for_diff)
            plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
            ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
            fig.suptitle(f"{task_name} | {cfg_label}", y=1.02, fontsize=12)
            fig.tight_layout()
            display(fig)
            plt.close(fig)
    else:
        display(Markdown("Diff outputs skipped: no diff parquet files found."))

    # 4) Four-panel projected trajectories: baseline vs scenario_3
    if "baseline" in unc_dfs and "scenario_3" in unc_dfs:
        fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=False)
        axes = axes.flatten()
        for ax, output in zip(axes, trajectory_outputs):
            pl.plot_two_scenarios(
                ax,
                unc_dfs,
                output,
                scenarios=["baseline", "scenario_3"],
                xlim=(2020, 2035),
                include_unc=True,
                ylab_fontsize=9,
            )
            ax.set_title(pl.title_lookup.get(output, output), fontsize=10)

        fig.suptitle(f"{task_name} | {cfg_label}", y=1.02, fontsize=12)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
    else:
        display(Markdown("Trajectories skipped: baseline or scenario_3 uncertainty file missing."))

In [ ]:
# Extract scenario_3 diff data across all completed tasks
scenario_3_data = []

for _, row in completed_df.iterrows():
    task_name = row["task"]
    task_path = Path(row["task_path"])
    cfg_label = row["config_label"]
    
    diff_file = task_path / "diff_quantiles_df_ref_baseline_scenario_3.parquet"
    if diff_file.exists():
        df_diff = pd.read_parquet(diff_file)
        scenario_3_data.append({
            "task": task_name,
            "config_label": cfg_label,
            "TB_averted_relative": df_diff.loc[0.5, "TB_averted_relative"],
            "TB_averted_relative_low": df_diff.loc[0.025, "TB_averted_relative"],
            "TB_averted_relative_high": df_diff.loc[0.975, "TB_averted_relative"],
        })

scenario_3_df = pd.DataFrame(scenario_3_data)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(scenario_3_df))
ax.errorbar(
    x,
    scenario_3_df["TB_averted_relative"],
    yerr=[
        scenario_3_df["TB_averted_relative"] - scenario_3_df["TB_averted_relative_low"],
        scenario_3_df["TB_averted_relative_high"] - scenario_3_df["TB_averted_relative"],
    ],
    fmt="o",
    color="#B22222",
    ecolor="#B22222",
    markersize=8,
    elinewidth=2.0,
    capsize=5,
)

ax.set_xticks(x)
ax.set_xticklabels(scenario_3_df["config_label"], rotation=45, ha="right")
ax.set_ylabel("TB episodes averted (relative %)")
ax.set_title("Scenario 3: % TB Episodes Averted Across All Completed Tasks")
ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
fig.tight_layout()
display(fig)
plt.close(fig)